## Workstream 2: Translate Pixels to Physical Flow Rates
## European Power Plant Validation — CEMF+IME+ERA5 Pipeline

**Project:** ECB/EIB Satellite Images for Emissions Tracking  
**Team:** Mihika Saraf, Yuhao Zhang  
**Date:** April 2026  
**Meeting:** Meeting 5 — Friday April 10, 2026  

---

### Workstream 2 Objective

This notebook implements the physical quantification layer of the methane detection pipeline.  
It takes CH4Net binary detection masks and converts them into physically meaningful emission  
rates with units, uncertainty bounds, and regulatory liability figures.

This is distinct from Workstream 1 (Vincent — model validation and detection) in the following way:

| Workstream | Question answered | Output |
|---|---|---|
| WS1 — Detection | Is there a plume? | S/C ratio, F1 score, TROPOMI cross-validation |
| WS2 — Quantification | How much methane is it emitting? | kg/h, annual tonnes, IRA liability |

### Scientific basis
- **CEMF**: Column-Enhanced Matched Filter — Varon et al. (2021), *Atmospheric Measurement Techniques*, 14, 2291–2307  
- **IME**: Integrated Mass Enhancement inversion — same reference  
- **ERA5 wind**: Irakulis-Loitxate et al. (2022), *ACS Earth and Space Chemistry*  
- **Uncertainty**: ±40% per Varon et al. (2021) for multispectral sensors  
- **IRA liability**: $1,500/tonne, 26 USC §136, 2026 statutory rate  

### Target site
Weisweiler lignite plant, Germany — confirmed detection by Workstream 1 (S/C = 23×  
above background). Using a confirmed detection site ensures the CEMF retrieval has  
real plume pixels to work with, producing a defensible physical emission rate.  
Fallback: Rybnik coal complex, Poland (TROPOMI-confirmed, S/C = 3.708).

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import shutil
import os
from google.colab import userdata

# ── Drive base path ────────────────────────────────────────────────────────────
# Update this shortcut ID if your Drive path differs
base = '/content/drive/.shortcut-targets-by-id/19xUy-jzTPqj83OSmQQAXyxJ2gmVw5tG6/EIB_2/Code & Data/CH4_Pipeline_Working'

# ── Copy model weights from Drive ─────────────────────────────────────────────
shutil.copy(f'{base}/models.py',      '/content/models.py')
shutil.copy(f'{base}/best_model.pth', '/content/best_model.pth')
print('Model files copied.')

# ── Write era5_utils.py locally ───────────────────────────────────────────────
era5_code = '''
import cdsapi
import xarray as xr
import numpy as np

def get_era5_wind(lat, lon, date):
    """
    Retrieve ERA5 10m wind speed for a given location and date.
    Returns wind speed in m/s. Falls back to 3.5 m/s on any failure.
    """
    try:
        c = cdsapi.Client()
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": [
                    "10m_u_component_of_wind",
                    "10m_v_component_of_wind"
                ],
                "year":  date[:4],
                "month": date[5:7],
                "day":   date[8:10],
                "time":  "12:00",
                "data_format": "netcdf",
                "download_format": "unarchived",
                "area": [lat + 0.25, lon - 0.25, lat - 0.25, lon + 0.25],
            },
            "era5.nc"
        )
        ds = xr.open_dataset("era5.nc")
        u  = float(ds["u10"].mean().values)
        v  = float(ds["v10"].mean().values)
        ds.close()
        speed = float((u**2 + v**2)**0.5)
        print(f"ERA5 retrieved: u={u:.4f} m/s, v={v:.4f} m/s, speed={speed:.4f} m/s")
        return speed
    except Exception as e:
        print(f"ERA5 query failed: {e}")
        print("Falling back to climatological default: 3.5 m/s")
        return 3.5
'''

with open('/content/era5_utils.py', 'w') as f:
    f.write(era5_code.strip())

print('era5_utils.py written.')
print('Contents of /content:', os.listdir('/content'))

Model files copied.
era5_utils.py written.
Contents of /content: ['.config', 'models.py', 'era5_utils.py', 'best_model.pth', 'drive', 'sample_data']


In [6]:
import subprocess

# ── Clone the GitHub repo using token from Colab Secrets ──────────────────────
# Add your token in Colab: left sidebar → key icon → "GITHUB_TOKEN"
github_token = userdata.get('GITHUB_TOKEN')

result = subprocess.run(
    ['git', 'clone',
     f'https://{github_token}@github.com/vincentcorn2/methane-emission-tracker.git',
     '/content/repo'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

if os.path.exists('/content/repo'):
    print('Repo cloned successfully.')
    print('Repo contents:', os.listdir('/content/repo'))
else:
    print('Clone failed — check GITHUB_TOKEN secret.')

# Add repo to Python path so we can import src modules
import sys
sys.path.insert(0, '/content/repo')
print('Repo added to Python path.')


Cloning into '/content/repo'...

Repo cloned successfully.
Repo contents: ['run_analysis.py', 'run_exhaustive_validation.py', 'tropomi_groningen.py', 'download_reference_tiles.py', 'extract_training_crops.py', 'apply_bitemporal_diff.py', 'requirements.txt', 'scripts', 'results_analysis', 'README.md', '.gitignore', 'tests', 'config', 'tropomi_mine_europe.py', 'HANDOFF.md', 'overnight_validation.py', 'run_expanded_sites.py', 'run_tropomi_colocation.py', 'src', 'approach_d_focal_dice.py', 'approach_b_rethreshold.py', '.git', 'docs', 'download_training_tiles.py', 'approach_a_centered_crops.py', 'run_quant_fixed.py', 'approach_c_retrain.py']
Repo added to Python path.


In [8]:
# Check what's actually defined in cemf.py
with open('/content/repo/src/quantification/cemf.py', 'r') as f:
    print(f.read())

import numpy as np
from dataclasses import dataclass
from typing import Optional

# Constants
DRY_AIR_COLUMN = 2.1e25        # molecules/m^2
MOL_WEIGHT_CH4 = 0.016         # kg/mol
AVOGADRO = 6.022e23
PIXEL_AREA_20M = 400.0         # m^2 at native 20m SWIR resolution

@dataclass
class CEMFResult:
    dxch4_map: np.ndarray          # per-pixel enhancement (ppb*m)
    plume_mask: np.ndarray         # binary mask at 20m
    total_mass_kg: float           # integrated column mass over plume
    scene_id: str
    timestamp: str
    pixel_area_m2: float = PIXEL_AREA_20M
    n_plume_pixels: int = 0
    retrieval_valid: bool = True

def run_cemf(
    b11: np.ndarray,
    b12: np.ndarray,
    mask: np.ndarray,
    scene_id: str,
    timestamp: str
) -> CEMFResult:
    """
    Translate Sentinel-2 B11/B12 TOA reflectance into per-pixel
    methane column enhancement (dXCH4) using a scene-derived
    background matched filter.

    Args:
        b11: Band 11 TOA reflectance at native 20m resoluti

In [9]:
with open('/content/repo/src/quantification/ime.py', 'r') as f:
    print(f.read())

"""
Emission Quantification: Pixel mask → Physical flow rate (kg/h)

Implements two complementary methodologies from the strategy document:

1. Integrated Mass Enhancement (IME) — primary estimate
   Total mass = sum of per-pixel CH4 enhancements over plume area
   Flow rate Q = (IME * U_eff) / L

2. Cross-Sectional Flux (CSF) — validation / uncertainty quantification
   Measures mass crossing orthogonal transects downwind of source
   Q = I(x) * U_eff

Both methods require effective wind speed (U_eff) from ERA5 reanalysis.

Key financial output:
  Flow rate in kg/h → annualize → apply IRA Waste Emissions Charge
  ($1,500/metric ton in 2026) → direct P&L impact for the emitter
"""
import logging
from dataclasses import dataclass
from typing import Optional

import numpy as np

logger = logging.getLogger(__name__)


@dataclass
class QuantificationResult:
    """Physical emission estimate with uncertainty bounds."""
    flow_rate_kgh: float            # primary estimate (kg CH4 / hour)
 

In [10]:
with open('/content/repo/src/quantification/emission_logger.py', 'r') as f:
    print(f.read())

"""
Time-series emission record logger.
Appends one JSON record per detection to a log file for
downstream climate scenario testing.
"""
import json
import os
from datetime import datetime
from typing import Optional


def log_emission_record(
    scene_id: str,
    timestamp: str,
    plume_centroid_lat: float,
    plume_centroid_lon: float,
    flow_rate_kgh: float,
    flow_rate_lower_kgh: float,
    flow_rate_upper_kgh: float,
    ch4net_peak_probability: float,
    cloud_cover_quality: str,
    wind_speed_ms: float,
    wind_source: str,
    n_plume_pixels: int,
    total_mass_kg: float,
    annual_tonnes: Optional[float] = None,
    ira_liability_usd: Optional[float] = None,
    log_path: str = "results/emission_timeseries.jsonl",
):
    """
    Append one emission record to the time-series log.
    Uses JSONL format (one JSON object per line) for easy streaming.
    """
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    record = {
        "logged_at": datetime.utcnow

In [12]:
import os

# Check what's in src/ingestion/
print('src/ingestion contents:')
for f in os.listdir('/content/repo/src/ingestion'):
    print(' ', f)

print()
print('src/quantification contents:')
for f in os.listdir('/content/repo/src/quantification'):
    print(' ', f)

src/ingestion contents:
  __init__.py
  copernicus_client.py
  __pycache__
  preprocessing.py

src/quantification contents:
  cemf.py
  emission_logger.py
  __init__.py
  __pycache__
  ime.py


In [13]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q cdsapi xarray netCDF4

# ── Write CDS API credentials ─────────────────────────────────────────────────
from google.colab import userdata
cds_key = userdata.get('CDS_API_KEY')
with open('/root/.cdsapirc', 'w') as f:
    f.write(f'url: https://cds.climate.copernicus.eu/api\nkey: {cds_key}\n')

print('CDS credentials written.')

# ── Verify all modules are importable ─────────────────────────────────────────
try:
    from src.quantification.cemf import run_cemf, CEMFResult
    from src.quantification.ime import CEMFIntegratedMassEnhancement, QuantificationResult
    from src.quantification.emission_logger import log_emission_record
    print('All Workstream 2 modules imported successfully:')
    print('  ✓ cemf.py            — run_cemf() → CEMFResult')
    print('  ✓ ime.py             — CEMFIntegratedMassEnhancement.estimate_from_cemf()')
    print('  ✓ emission_logger.py — log_emission_record()')
    print()
    print('Note: ERA5 wind retrieved via era5_utils.py (Section 8)')
except ImportError as e:
    print(f'Import error: {e}')

CDS credentials written.
All Workstream 2 modules imported successfully:
  ✓ cemf.py            — run_cemf() → CEMFResult
  ✓ ime.py             — CEMFIntegratedMassEnhancement.estimate_from_cemf()
  ✓ emission_logger.py — log_emission_record()

Note: ERA5 wind retrieved via era5_utils.py (Section 8)


In [14]:
# ── Target site definition ────────────────────────────────────────────────────
# Primary: Weisweiler — confirmed by WS1 with S/C = 23x above background
TARGET_SITE = {
    "name":    "Weisweiler Lignite Plant",
    "lat":     50.828,
    "lon":     6.318,
    "country": "DE",
    "fuel":    "lignite",
    "mw":      2400,
    "ws1_sc_ratio": 23.0,    # Signal/Control ratio from Workstream 1
    "tropomi_confirmed": True
}

# Fallback — uncomment if Weisweiler has no usable scene
# TARGET_SITE = {
#     "name":    "Rybnik Coal Complex",
#     "lat":     50.097,
#     "lon":     18.541,
#     "country": "PL",
#     "fuel":    "coal",
#     "mw":      1750,
#     "ws1_sc_ratio": 3.708,
#     "tropomi_confirmed": True
# }

print('Target site configured:')
for k, v in TARGET_SITE.items():
    print(f'  {k}: {v}')

Target site configured:
  name: Weisweiler Lignite Plant
  lat: 50.828
  lon: 6.318
  country: DE
  fuel: lignite
  mw: 2400
  ws1_sc_ratio: 23.0
  tropomi_confirmed: True


In [23]:
import json

# ── Load Workstream 1 quantification results ──────────────────────────────────
# These were computed by Vincent using the full CEMF+IME pipeline
# on confirmed European detections. We use them as our WS2 inputs.

quant_path = '/content/repo/results_analysis/quantification.json'

with open(quant_path) as f:
    quant_results = json.load(f)

print('European quantification results from repo:')
print(json.dumps(quant_results, indent=2))

European quantification results from repo:
[
  {
    "site": "groningen",
    "methodology": "CEMF+IME",
    "wind_ms": 5.5,
    "flow_rate_kgh": 28.7591,
    "flow_rate_lower_kgh": 17.2555,
    "flow_rate_upper_kgh": 40.2628,
    "annual_tonnes": 251.9299,
    "ira_usd": 377894.9025,
    "plume_pixels": 1980,
    "cemf_valid": true,
    "total_mass_kg": 15.0041,
    "cemf_sensitivity_coeff": "4e-7 (Varon 2021, AMT Sec 2.2)"
  },
  {
    "site": "maasvlakte",
    "methodology": "CEMF+IME",
    "wind_ms": 4.5,
    "flow_rate_kgh": 426.4968,
    "flow_rate_lower_kgh": 255.8981,
    "flow_rate_upper_kgh": 597.0956,
    "annual_tonnes": 3736.1122,
    "ira_usd": 5604168.2805,
    "plume_pixels": 1565,
    "cemf_valid": true,
    "total_mass_kg": 121.3673,
    "cemf_sensitivity_coeff": "4e-7 (Varon 2021, AMT Sec 2.2)"
  }
]


In [24]:
from era5_utils import get_era5_wind
from src.quantification.emission_logger import log_emission_record
import os

os.makedirs('/content/results', exist_ok=True)
jsonl_path = '/content/results/emission_timeseries.jsonl'

# Site metadata to pair with quantification results
site_metadata = {
    "groningen": {
        "lat": 53.252, "lon": 6.682,
        "date": "2024-06-28",
        "scene_id": "S2_Groningen_20240628",
        "country": "NL", "fuel": "gas_field",
        "ws1_sc_ratio": 4.191,
        "tropomi_confirmed": False,
        "tropomi_note": "TROPOMI non-detection confirmed — signal is nl_grijpskerk compressor station on same tile"
    },
    "maasvlakte": {
        "lat": 51.954, "lon": 4.025,
        "date": "2024-06-28",
        "scene_id": "S2_Maasvlakte_20240628",
        "country": "NL", "fuel": "gas_terminal",
        "ws1_sc_ratio": None,
        "tropomi_confirmed": True,
        "tropomi_note": "TROPOMI confirmed active emitter"
    }
}

ws2_records = []

for result in quant_results:
    site_key  = result['site']
    meta      = site_metadata.get(site_key, {})
    lat       = meta.get('lat')
    lon       = meta.get('lon')
    date_str  = meta.get('date', '2024-06-28')

    print(f'\nProcessing: {site_key}')
    print(f'  CEMF mass:     {result["total_mass_kg"]:.4f} kg')
    print(f'  Plume pixels:  {result["plume_pixels"]}')
    print(f'  CEMF valid:    {result["cemf_valid"]}')

    # ── ERA5 wind for this site ───────────────────────────────────────────────
    if lat and lon:
        wind = get_era5_wind(lat, lon, date_str)
        wind_source = 'ERA5_reanalysis' if wind != 3.5 else 'climatological_fallback_3.5ms'
    else:
        wind = result.get('wind_ms', 3.5)
        wind_source = 'from_quantification_json'

    print(f'  Wind:          {round(wind, 4)} m/s ({wind_source})')

    # ── Use Vincent's flow rate but with our ERA5 wind ────────────────────────
    # If our ERA5 wind differs from Vincent's, recompute Q
    # Q = (mass * wind) / plume_length
    # Vincent used wind_ms from quantification.json
    vincent_wind = result.get('wind_ms', 3.5)
    vincent_q    = result.get('flow_rate_kgh', 0)

    if vincent_wind > 0 and vincent_q > 0:
        # Back-calculate plume length from Vincent's result
        plume_length_m = (result['total_mass_kg'] * vincent_wind) / (vincent_q / 3600)
        # Recompute with our ERA5 wind
        Q_kg_per_h = (result['total_mass_kg'] * wind) / plume_length_m * 3600
    else:
        Q_kg_per_h   = vincent_q
        plume_length_m = 0

    Q_lower              = round(Q_kg_per_h * 0.60, 4)
    Q_upper              = round(Q_kg_per_h * 1.40, 4)
    annual_tonnes        = round(Q_kg_per_h * 8760 / 1000, 2)
    ira_waste_charge_usd = round(annual_tonnes * 1500, 2)

    print(f'  Q (WS2 ERA5):  {round(Q_kg_per_h, 2)} kg/h')
    print(f'  Range:         {Q_lower} – {Q_upper} kg/h')
    print(f'  Annual tonnes: {annual_tonnes} t/yr')
    print(f'  IRA liability: ${ira_waste_charge_usd:,.0f}')

    # ── Log to JSONL ──────────────────────────────────────────────────────────
    record = log_emission_record(
        scene_id=meta.get('scene_id', site_key),
        timestamp=date_str + "T00:00:00Z",
        plume_centroid_lat=lat or 0.0,
        plume_centroid_lon=lon or 0.0,
        flow_rate_kgh=round(float(Q_kg_per_h), 4),
        flow_rate_lower_kgh=Q_lower,
        flow_rate_upper_kgh=Q_upper,
        ch4net_peak_probability=0.0,
        cloud_cover_quality="Optimal",
        wind_speed_ms=round(float(wind), 4),
        wind_source=wind_source,
        n_plume_pixels=result['plume_pixels'],
        total_mass_kg=round(float(result['total_mass_kg']), 6),
        annual_tonnes=annual_tonnes,
        ira_liability_usd=ira_waste_charge_usd,
        log_path=jsonl_path,
    )

    # Add extra fields
    record['site_name']         = site_key
    record['country']           = meta.get('country', '')
    record['fuel_type']         = meta.get('fuel', '')
    record['ws1_sc_ratio']      = meta.get('ws1_sc_ratio')
    record['tropomi_confirmed'] = meta.get('tropomi_confirmed')
    record['tropomi_note']      = meta.get('tropomi_note', '')
    record['cemf_valid']        = result['cemf_valid']
    record['cemf_sensitivity']  = result.get('cemf_sensitivity_coeff', '')
    record['vincent_wind_ms']   = vincent_wind
    record['ws2_era5_wind_ms']  = round(float(wind), 4)
    record['workstream']        = 'WS2 — Translate Pixels to Physical Flow Rates'
    record['plume_length_m']    = round(float(plume_length_m), 1)

    # Rewrite last line with extra fields
    lines = open(jsonl_path).readlines()
    lines[-1] = json.dumps(record) + '\n'
    with open(jsonl_path, 'w') as f:
        f.writelines(lines)

    ws2_records.append(record)

print(f'\n\nAll records written to: {jsonl_path}')
print(f'Total records: {len(ws2_records)}')


Processing: groningen
  CEMF mass:     15.0041 kg
  Plume pixels:  1980
  CEMF valid:    True


2026-04-10 06:34:28,273 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

5c7541995e8ddf746a5a8b6029eeb36f.nc:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

ERA5 retrieved: u=7.7815 m/s, v=2.4667 m/s, speed=8.1631 m/s
  Wind:          8.1631 m/s (ERA5_reanalysis)
  Q (WS2 ERA5):  42.68 kg/h
  Range:         25.6105 – 59.7579 kg/h
  Annual tonnes: 373.91 t/yr
  IRA liability: $560,865

Processing: maasvlakte
  CEMF mass:     121.3673 kg
  Plume pixels:  1565
  CEMF valid:    True


2026-04-10 06:34:53,827 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

bd56f25f800205aebb29c0d7a227e141.nc:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

ERA5 retrieved: u=5.9609 m/s, v=1.5697 m/s, speed=6.1641 m/s
  Wind:          6.1641 m/s (ERA5_reanalysis)
  Q (WS2 ERA5):  584.22 kg/h
  Range:         350.532 – 817.908 kg/h
  Annual tonnes: 5117.77 t/yr
  IRA liability: $7,676,655


All records written to: /content/results/emission_timeseries.jsonl
Total records: 2


In [25]:
print('=' * 65)
print('WORKSTREAM 2 — EUROPEAN VALIDATION SUMMARY')
print('Translate Pixels to Physical Flow Rates')
print('=' * 65)
print()

for r in ws2_records:
    print(f'Site:                  {r["site_name"]}')
    print(f'Country:               {r["country"]}')
    print(f'Fuel type:             {r["fuel_type"]}')
    print(f'TROPOMI confirmed:     {r["tropomi_confirmed"]}')
    if r.get("tropomi_note"):
        print(f'TROPOMI note:          {r["tropomi_note"]}')
    print(f'Plume pixels:          {r["n_plume_pixels"]}')
    print(f'Total mass:            {r["total_mass_kg"]} kg')
    print(f'Wind (Vincent):        {r["vincent_wind_ms"]} m/s')
    print(f'Wind (WS2 ERA5):       {r["ws2_era5_wind_ms"]} m/s ({r["wind_source"]})')
    print(f'Q (flow rate):         {r["q_cemf_kg_per_hour"]} kg/h')
    print(f'Q range (±40%):        {r["q_lower_kg_per_hour"]} – {r["q_upper_kg_per_hour"]} kg/h')
    print(f'Annual tonnes:         {r["annual_tonnes_if_continuous"]} t/yr')
    print(f'IRA liability (2026):  ${r["ira_waste_charge_usd_2026"]:,.0f}')
    print(f'Methodology:           CEMF+IME (Varon et al. 2021)')
    print(f'CEMF sensitivity:      {r["cemf_sensitivity"]}')
    print()
    print('-' * 65)
    print()

print('Uncertainty: ±40% per Varon et al. (2021) multispectral retrieval')
print('IRA rate:    $1,500/tonne (26 USC §136, 2026 statutory rate)')
print('Annual:      Assumes continuous emission (upper-bound liability)')

WORKSTREAM 2 — EUROPEAN VALIDATION SUMMARY
Translate Pixels to Physical Flow Rates

Site:                  groningen
Country:               NL
Fuel type:             gas_field
TROPOMI confirmed:     False
TROPOMI note:          TROPOMI non-detection confirmed — signal is nl_grijpskerk compressor station on same tile
Plume pixels:          1980
Total mass:            15.0041 kg
Wind (Vincent):        5.5 m/s
Wind (WS2 ERA5):       8.1631 m/s (ERA5_reanalysis)
Q (flow rate):         42.6842 kg/h
Q range (±40%):        25.6105 – 59.7579 kg/h
Annual tonnes:         373.91 t/yr
IRA liability (2026):  $560,865
Methodology:           CEMF+IME (Varon et al. 2021)
CEMF sensitivity:      4e-7 (Varon 2021, AMT Sec 2.2)

-----------------------------------------------------------------

Site:                  maasvlakte
Country:               NL
Fuel type:             gas_terminal
TROPOMI confirmed:     True
TROPOMI note:          TROPOMI confirmed active emitter
Plume pixels:          1565
Total 